GSE114725 — Data loading

Loads the raw CSV supplementary file (downloaded from GEO as GSE114725_rna_raw.csv.gz, extracted to raw_corrected.csv), constructs an AnnData object, and saves it as GSE114725_raw.h5ad for use in downstream QC (03_phase1_QC_V2.ipynb). Must be run within the activated scrna conda environment.

In [2]:
import pandas as pd
import scanpy as sc
from scipy import sparse
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"

file_path = RAW_DIR / "GSE114725_rna_raw.csv" / "raw_corrected.csv"
meta_cols = ["patient", "tissue", "replicate", "cluster", "cellid"]

In [3]:
# Load metadata columns and expression values separately, expression
# in chunks (file is ~2.8GB; avoids loading the full dense CSV at once)
obs = pd.read_csv(file_path, usecols=meta_cols)

chunks = []
for chunk in pd.read_csv(file_path, chunksize=2000):
    expr = chunk.drop(columns=meta_cols)
    chunks.append(sparse.csr_matrix(expr.values))
X = sparse.vstack(chunks)

print(f"Expression matrix: {X.shape}")
print(f"Metadata: {obs.shape}")

Expression matrix: (47016, 14875)
Metadata: (47016, 5)


In [4]:
adata = sc.AnnData(X)
adata.obs = obs.set_index("cellid").rename_axis("cell_id")
adata.obs.index = adata.obs.index.astype(str)
adata.var_names = expr.columns

adata.obs_names_make_unique()
adata.var_names_make_unique()

print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster'
        patient tissue  replicate  cluster
cell_id                                   
246         BC5  TUMOR          1        2
260         BC5  TUMOR          1        2
346         BC5  TUMOR          1        2
188         BC5  TUMOR          1        4
33          BC5  TUMOR          1        5


C:\Users\annam\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
adata.write_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
print(f"Saved: {RAW_DIR / 'GSE114725_raw.h5ad'}")

Saved: C:\Users\annam\Dissertation 2026\Data\Raw\GSE114725_raw.h5ad
